In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt


# Garante que o pacote `src` seja importável mesmo se o Jupyter estiver com cwd em `notebooks`
_cwd = Path().resolve()
_repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from src.config import get_paths
from src.io import load_base_disparo, load_dim_telefone
from src.metrics import delivered_flag, aggregate_rate_with_ci
from src.scoring import SystemScoreParams, build_system_ranking, score_phones_for_cpf

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(30)
pl.Config.set_fmt_float("full")

paths = get_paths(_repo_root)
paths

Paths(repo_root=WindowsPath('C:/Users/Nicole Pesoa/OneDrive/Desktop/desafio-tecnico/desafio-cientista-dados-pleno-campanhas'), data_raw=WindowsPath('C:/Users/Nicole Pesoa/OneDrive/Desktop/desafio-tecnico/desafio-cientista-dados-pleno-campanhas/data/raw'), data_processed=WindowsPath('C:/Users/Nicole Pesoa/OneDrive/Desktop/desafio-tecnico/desafio-cientista-dados-pleno-campanhas/data/processed'))

In [2]:
# 1. Carrega os dados brutos e o dataset processado no Notebook 1
events_df = pl.read_parquet(paths.data_processed / "events_enriched.parquet")
dim_lf = load_dim_telefone(paths.data_raw)

# 2. Recria a tabela dim_aparicoes (necessária para extrair os candidatos por CPF)
dim_aparicoes = (
    dim_lf
    .with_columns(telefone_mascarado=pl.col("telefone_numero"))
    .select(
        "telefone_numero",
        "telefone_ddi",
        "telefone_ddd",
        "telefone_tipo",
        "telefone_nacionalidade",
        "telefone_qualidade",
        "telefone_aparicoes",
        "telefone_aparicoes_quantidade",
        "telefone_proprietarios_quantidade",
        "telefone_sistemas_quantidade",
        "validacao_telefone",
    )
    .explode("telefone_aparicoes")
    .unnest("telefone_aparicoes")
    .rename(
        {
            "telefone_numero": "telefone_mascarado",
            "id_sistema": "id_sistema_mask",
            "registro_data_atualizacao": "registro_data_atualizacao",
            "cpf": "cpf_aparicao",
        }
    )
)

# 3. Recria a tabela events_decay (necessária para calcular o Ranking de Sistemas)
events_decay = (
    events_df.filter(pl.col("age_days").is_not_null())
    .with_columns(age_days=pl.when(pl.col("age_days") < 0).then(0.0).otherwise(pl.col("age_days")))
)

# 4. Define o parâmetro de decaimento descoberto na Análise Exploratória (Notebook 2)
# Inserir o valor calculado previamente traz contexto de negócio à modelagem
half_life_odds = 507.065

## Parte 2.1 Ranking de sistemas

Construção do score

Para cada sistema

1. Contar tentativas \(n\) e entregas \(s\)
2. Estimar uma taxa suavizada via média a posteriori Beta

\[\hat p = \frac{s + \alpha}{n + \alpha + \beta}\]

Isso reduz instabilidade em sistemas com baixo volume.

Depois aplico um encolhimento adicional por volume para reduzir ranking inflado por amostras pequenas

\[score = \hat p \cdot \frac{n}{n + k}\]

com \(k\) constante de regularização.

In [3]:
system_rank = build_system_ranking(
    df_events=events_decay.drop_nulls(["id_sistema_mask"]),
    system_col="id_sistema_mask",
    delivered_col="delivered",
    age_days_col="age_days",
    params=SystemScoreParams(prior_alpha=2.0, prior_beta=2.0, half_life_days=float(half_life_odds) if np.isfinite(half_life_odds) else 365.0),
)

system_rank.head(30)

id_sistema_mask,n,successes,mean_age_days,posterior_mean,score
str,u32,i64,f64,f64,f64
"""-4704067261970591609""",116239,114624,124.63212863152643,0.986089484958234,0.9818660057226819
"""3094574413675758272""",154881,144756,253.07526423512246,0.9346160054233786,0.9316085077067229
"""1257277410380486863""",52225,46385,665.1138343705122,0.8881464320588179,0.8797239907875157
"""-2757366171786647144""",2476,2340,328.16033925686594,0.9443548387096774,0.7856930714533472
"""4458959843028638627""",2832,2457,319.51871468926555,0.8670662905500706,0.7369543021722088
"""-133612832286195827""",2755,2303,779.2656987295826,0.8354476259514316,0.7071146572952977


## Parte 2.2 Algoritmo de escolha dos dois melhores telefones por CPF

Objetivo

Dado um CPF com múltiplos telefones candidatos, retornar os dois com maior probabilidade de entrega.

Score por telefone

\[final\_score = system\_score \cdot recency\_weight(age\_days) \cdot ddd\_factor\]

Onde

1. `system_score` vem do ranking anterior
2. `recency_weight` usa decaimento exponencial com meia vida em dias

\[recency\_weight = 2^{-age\_days / half\_life}\]

3. `ddd_factor` é um ajuste opcional por desempenho histórico do DDD ou por validações estruturais

Decisão

Ordenar os telefones pelo `final_score` e escolher os dois maiores

In [4]:
ddd_rates = (
    events_df.filter(pl.col("telefone_ddd").is_not_null())
    .group_by("telefone_ddd")
    .agg(n=pl.len(), rate=pl.col("delivered").mean())
    .filter(pl.col("n") >= 200)
    .sort("rate", descending=True)
)

q_low = float(ddd_rates.get_column("rate").quantile(0.2)) if ddd_rates.height else 0.0
q_high = float(ddd_rates.get_column("rate").quantile(0.8)) if ddd_rates.height else 0.0

bonus_map = {}
for row in ddd_rates.select(["telefone_ddd", "rate"]).to_dicts():
    ddd = str(row["telefone_ddd"])
    r = float(row["rate"])
    if r >= q_high:
        bonus_map[ddd] = 0.05
    elif r <= q_low:
        bonus_map[ddd] = -0.05

params = SystemScoreParams(
    prior_alpha=2.0,
    prior_beta=2.0,
    half_life_days=float(half_life_odds) if np.isfinite(half_life_odds) else 365.0,
    ddd_bonus=bonus_map,
)

len(bonus_map), list(bonus_map.items())[:5]

(4,
 [('-1181433720517268842', 0.05),
  ('2685248380958544781', 0.05),
  ('1110362451252208393', -0.05),
  ('-4523905025782287081', -0.05)])

In [5]:
dim_candidates = (
    dim_aparicoes
    .select(
        cpf=pl.col("cpf_aparicao"),
        telefone=pl.col("telefone_mascarado"),
        id_sistema_mask=pl.col("id_sistema_mask"),
        registro_data_atualizacao=pl.col("registro_data_atualizacao"),
        telefone_ddd=pl.col("telefone_ddd"),
    )
    .drop_nulls(["cpf", "telefone", "id_sistema_mask", "registro_data_atualizacao"])
)

cpf_exemplo = int(dim_candidates.select("cpf").limit(1).collect().item())

cand_df = dim_candidates.filter(pl.col("cpf") == cpf_exemplo).collect()

now_ts = pd.Timestamp("2026-04-27")

scored = score_phones_for_cpf(
    df_candidates=cand_df,
    system_rank=system_rank,
    system_col="id_sistema_mask",
    phone_col="telefone",
    updated_at_col="registro_data_atualizacao",
    ddd_col="telefone_ddd",
    now_ts=now_ts,
    params=params,
)

scored.head(10)

telefone,id_sistema_mask,registro_data_atualizacao,telefone_ddd,final_score,rank
i64,str,date,i64,f64,i64
-6862804366069381626,"""1257277410380486863""",2024-10-30,-1181433720517268842,0.4391152150867542,1


In [6]:
cpfs_multi = (
    dim_candidates
    .group_by("cpf")
    .agg(n_telefones=pl.col("telefone").n_unique())
    .filter(pl.col("n_telefones") >= 3)
    .sort("n_telefones", descending=True)
    .collect()
)

cpf_exemplo = int(cpfs_multi[0, "cpf"])
cpf_exemplo

5439497641908800296

In [7]:
scored.filter(pl.col("rank") <= 2)

telefone,id_sistema_mask,registro_data_atualizacao,telefone_ddd,final_score,rank
i64,str,date,i64,f64,i64
-6862804366069381626,"""1257277410380486863""",2024-10-30,-1181433720517268842,0.4391152150867542,1


In [8]:
cand_df = dim_candidates.filter(pl.col("cpf") == cpf_exemplo).collect()

cand_df.height, cand_df.select(pl.col("telefone").n_unique()).item()

(398, 398)

## Notas de execução e troubleshooting

### Import do pacote src

Se aparecer `ModuleNotFoundError: No module named 'src'`, o motivo usual é o Jupyter com diretório atual em `notebooks`. Este Notebook injeta o caminho do repositório no `sys.path` na primeira célula de código. Se você executou fora de ordem, reinicie o kernel e rode tudo.

### Barras de erro negativas em gráficos

O Matplotlib não aceita `xerr` ou `yerr` negativos. As células de gráfico saturam os erros em zero e tratam valores ausentes para garantir execução reprodutível.

### Estabilidade numérica no decaimento

Em bases grandes, regressão logística linha a linha pode sofrer instabilidade. A etapa de decaimento agrega por bins e usa padronização para melhorar estabilidade. O objetivo é interpretar a tendência de decaimento e produzir um parâmetro operacional de meia vida.

In [9]:
scored = score_phones_for_cpf(
    df_candidates=cand_df,
    system_rank=system_rank,
    system_col="id_sistema_mask",
    phone_col="telefone",
    updated_at_col="registro_data_atualizacao",
    ddd_col="telefone_ddd",
    now_ts=pd.Timestamp("2026-04-27"),
    params=params,
)

scored, scored.filter(pl.col("rank") <= 2)

(shape: (398, 6)
 ┌──────────────────────┬──────────────────────┬──────────────────────┬─────────────────────┬────────────────────┬──────┐
 │ telefone             ┆ id_sistema_mask      ┆ registro_data_atuali ┆ telefone_ddd        ┆ final_score        ┆ rank │
 │ ---                  ┆ ---                  ┆ zacao                ┆ ---                 ┆ ---                ┆ ---  │
 │ i64                  ┆ str                  ┆ ---                  ┆ i64                 ┆ f64                ┆ i64  │
 │                      ┆                      ┆ date                 ┆                     ┆                    ┆      │
 ╞══════════════════════╪══════════════════════╪══════════════════════╪═════════════════════╪════════════════════╪══════╡
 │ -117014160379137149  ┆ -4704067261970591609 ┆ 2025-11-15           ┆ -118143372051726884 ┆ 0.8250363485490838 ┆ 1    │
 │                      ┆                      ┆                      ┆ 2                   ┆                    ┆      │
 │ 2516

In [10]:
scored.filter(pl.col("rank") <= 10).select("telefone").n_unique()

10

In [11]:
top2_telefones = (
    scored
    .group_by("telefone")
    .agg(
        final_score=pl.col("final_score").max(),
        registro_data_atualizacao=pl.col("registro_data_atualizacao").max(),
        id_sistema_mask=pl.col("id_sistema_mask").first(),
        telefone_ddd=pl.col("telefone_ddd").first(),
    )
    .sort("final_score", descending=True)
    .head(2)
)

top2_telefones

telefone,final_score,registro_data_atualizacao,id_sistema_mask,telefone_ddd
i64,f64,date,str,i64
2516516057539296640,0.8250363485490838,2025-11-15,"""-4704067261970591609""",-1181433720517268842
-117014160379137149,0.8250363485490838,2025-11-15,"""-4704067261970591609""",-1181433720517268842


In [12]:
top2 = scored.filter(pl.col("rank") <= 2)

top2.select(
    pl.col("telefone").n_unique().alias("telefones_distintos"),
    pl.len().alias("linhas_top2"),
)

telefones_distintos,linhas_top2
u32,u32
2,2


## Parte 3 Desenho de experimento A B

Objetivo

Validar se a estratégia de priorização melhora entrega e reduz custo por entrega em comparação com a estratégia atual de baseline.

Unidade de randomização

CPF para evitar interferência entre telefones do mesmo cidadão.

Tratamentos

Controle

Selecionar aleatoriamente dois telefones elegíveis do CPF

Tratamento

Selecionar os dois telefones com maior `final_score`

Hipótese nula

\(H_0\) a taxa de entrega do tratamento é menor ou igual à do controle

\(p_T \le p_C\)

Hipótese alternativa

\(H_1\) a taxa de entrega do tratamento é maior

\(p_T > p_C\)

Métrica primária

Taxa de DELIVERED por CPF ao menos uma entrega nas duas tentativas

Métricas secundárias

1. Taxa de DELIVERED por tentativa
2. Custo por entrega proxy \(2\) tentativas dividido por entregas
3. Taxa de falhas por motivo `descricao_falha` quando disponível
4. Tempo até resposta quando existir

Guardrails

1. Taxa de bloqueio ou padrão suspeito se existir via validação
2. Reclamação ou opt out se houver no produto

In [13]:
from math import ceil
from scipy.stats import norm


def sample_size_two_proportions(p_c: float, p_t: float, alpha: float = 0.05, power: float = 0.8) -> int:
    if not (0 < p_c < 1 and 0 < p_t < 1):
        return 0
    z_alpha = float(norm.ppf(1 - alpha))
    z_beta = float(norm.ppf(power))
    p_bar = (p_c + p_t) / 2

    num = (z_alpha * np.sqrt(2 * p_bar * (1 - p_bar)) + z_beta * np.sqrt(p_c * (1 - p_c) + p_t * (1 - p_t))) ** 2
    den = (p_t - p_c) ** 2
    return int(ceil(num / den))


baseline_p = float(events_df.select(pl.col("delivered").mean()).item())

desired_abs_lift = 0.01  # exemplo: +1 ponto percentual
p_t = min(0.99, baseline_p + desired_abs_lift)

n_per_group = sample_size_two_proportions(baseline_p, p_t, alpha=0.05, power=0.8)

baseline_p, p_t, n_per_group

(0.9085057810603149, 0.9185057810603149, 9769)

### Duração estimada

Dado um volume diário \(V\) de CPFs elegíveis por dia, a duração aproximada em dias é

\[dias \approx \frac{2 \cdot n\_per\_group}{V}\]

O fator \(2\) é porque precisamos preencher controle e tratamento.

Recomendação operacional

1. Rodar de uma a duas semanas no mínimo para capturar ciclos de uso do WhatsApp e sazonalidade de dia da semana
2. Monitorar diariamente as métricas primárias e guardrails
3. Congelar a regra do ranking durante o teste para evitar contaminação